# Mastering the Database Class

# Introduction

The ``Database`` class is the orchestrator of ``sqlalchemyobjects``. It manages the database connection, the engine, and the collection of table manifestations. While you can use it directly, the recommended pattern is to inherit from it to create a strongly-typed, application-specific database class.

This tutorial covers:
- Defining a custom Database class
- Managing connections (Open/Close) and Context Managers
- Creating Sessions (Sync and Async)
- Understanding ``table_map``
- Handling Database creation

**Prerequisites:**
- Basic familiarity with Python and SQLAlchemy
- Installed package: ``sqlalchemyobjects``

## Table of Contents

- [Importing the Module](#Importing-the-Module)
- [Defining a Custom Database](#Defining-a-Custom-Database)
- [Initialization and Creation](#Initialization-and-Creation)
- [Connection Management](#Connection-Management)
- [Session Management](#Session-Management)
- [Async Database](#Async-Database)
- [API Highlights](#API-Highlights)
- [Conclusion](#Conclusion)



# Importing the Module



In [ ]:
from pathlib import Path
from sqlalchemy.orm import DeclarativeBase, Mapped
from sqlalchemy.ext.asyncio import AsyncAttrs

from sqlalchemyobjects import Database, BaseTableSchema, TableManifestation

# Defining a Custom Database

The Unified Workflow suggests creating a subclass of ``Database``. This allows you to define the schema and register tables in one place.

First, let's define a simple table for this example.



In [ ]:
# 1. Table Schema
class ItemSchema(BaseTableSchema):
    """Schema definition for the Item table, specifying columns."""
    __tablename__ = "items_default"
    name: Mapped[str]

class ItemManifestation(TableManifestation):
    """Manifestation class for handling Item table operations."""

# 2. Database Schema
class DatabaseSchema(AsyncAttrs, DeclarativeBase):
    """Declarative base class for the database schema, supporting async attributes."""

class ItemTable(ItemSchema, DatabaseSchema):
    """SQLAlchemy table definition for Items, combining the schema and database base."""
    __tablename__ = "items"

# 3. Database Class
class AppDatabase(Database):
    """Custom Database class orchestrating the connection and table manifestations."""
    # Bind the SQLAlchemy Base Schema
    schema = DatabaseSchema

    # Register tables: Name -> (ManifestationClass, TableClass, Kwargs) The Kwargs are passed to the ManifestationClass constructor when it is instantiated.
    table_map = {
        "items": (ItemManifestation, ItemTable, {})
    }

    # Property for easy access (Optional)
    @property
    def items(self) -> ItemManifestation:
        return self.tables["items"]

# Initialization and Creation

When initializing the database, you provide the path. You must also ensure the tables are created in the file.



In [ ]:
db_path = Path("tutorial_database.sqlite")
if db_path.exists():
    db_path.unlink()

# Instantiate
db = AppDatabase(path=db_path)

# Create tables
# This creates the engine and issues CREATE TABLE statements
db.create_database()

print(f"Database created at {db_path}")

# Connection Management

The ``Database`` object manages the SQLAlchemy Engine.

## Explicit Open/Close

You can manually open and close the connection.



In [ ]:
db.open()
print(f"Is open? {db.is_open}")

# Perform operations...

db.close()
print(f"Is open? {db.is_open}")

## Context Manager (Recommended)

Using the database as a context manager ensures it closes automatically.



In [ ]:
with AppDatabase(path=db_path) as database:
    print(f"Inside context: Is open? {database.is_open}")
    # Database is ready to use
    database.items.insert({"name": "Context Item"})

print(f"Outside context: Is open? {database.is_open}")

# Session Management

The ``Database`` class provides a factory for SQLAlchemy Sessions. This ensures sessions are bound to the correct engine.



In [ ]:
db.open()

# creating a session
with db.create_session() as session:
    # This is a standard SQLAlchemy Session
    # You can use it for custom queries
    print("Session created.")

    # Example: Direct SQLAlchemy usage
    items = session.query(ItemTable).all()
    print(f"Items in DB: {len(items)}")

db.close()

# Async Database

The ``Database`` class handles asynchronous engines seamlessly. You just need to set ``async_engine=True``.



In [ ]:
import anyio

async def async_db_demo():
    """Demonstration of using the database in asynchronous mode."""
    async_path = anyio.Path("tutorial_database_async.sqlite")
    if await async_path.exists():
        await async_path.unlink()

    # Initialize with async_engine=True
    # We use the SAME AppDatabase class!
    async_db = AppDatabase(path=str(async_path), async_engine=True)

    # Create tables asynchronously
    await async_db.create_database_async()

    # Async Context Manager
    async with async_db as adb:
        print(f"Async DB Open? {adb.is_open}")

        # Create Async Session
        async with adb.create_async_session() as session:
            print("Async session ready.")
            # Perform async operations...
            await adb.items.insert_async({"name": "Async Item"})

    await async_path.unlink()

await async_db_demo()

In [ ]:
# Cleanup
if db_path.exists():
    db_path.unlink()

# API Highlights

- **``Database(path, schema, table_map, async_engine)``**:
    - **``path``**: Path to the database file (str or Path).
    - **``schema``**: The SQLAlchemy DeclarativeBase.
    - **``table_map``**: Dictionary mapping names to table definitions.
    - **``create_database()`` / ``create_database_async()``**: Creates tables.
    - **``create_session()`` / ``create_async_session()``**: Yields a session.
    - **``tables``**: Dictionary of instantiated Manifestations.



# Conclusion

The ``Database`` class is the central hub of your data layer. By subclassing it, you create a clean, organized entry point for your application's data access.

- **Reference**: See ``docs/concepts/database.rst``.

